# ЛР 01.2 — Wrapper + Embedded методы отбора признаков (TODO)

Ориентир по времени: 90 минут.

## Цель
Сравнить wrapper и embedded подходы, затем собрать 2-3 кандидатных feature set для итогового сравнения моделей.

In [ ]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE, SequentialFeatureSelector
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression

ROOT = Path('..') if (Path('..') / 'lab_utils.py').exists() else Path('.')
ROOT = ROOT.resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from lab_utils import (
    load_dataset,
    split_xy,
    train_test_split_stratified,
    build_preprocessor,
    transform_with_names,
    to_dense,
    append_ranking_rows,
    build_shortlist,
)

SEED = 42
DATASETS = {
    'medical': ROOT / 'data' / 'medical_cardiovascular_risk.csv',
    'finance': ROOT / 'data' / 'finance_credit_risk.csv',
}
OUTPUT_DIR = ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

## Подготовка данных и загрузка shortlist из Notebook 1
Если `shortlist_filter.json` уже есть, ограничиваем пространство поиска для ускорения wrapper-процедур.

In [ ]:
filter_shortlist_path = OUTPUT_DIR / 'shortlist_filter.json'
if filter_shortlist_path.exists():
    with open(filter_shortlist_path, 'r', encoding='utf-8') as f:
        filter_shortlists = json.load(f)
else:
    filter_shortlists = {}

print('shortlist loaded:', bool(filter_shortlists))

In [ ]:
prepared = {}
for dataset_name, path in DATASETS.items():
    df = load_dataset(str(path))
    X, y = split_xy(df)
    X_train, X_test, y_train, y_test = train_test_split_stratified(X, y, random_state=SEED)

    preprocessor = build_preprocessor(X_train)
    X_train_t, X_test_t, feature_names = transform_with_names(preprocessor, X_train, X_test)

    X_train_t = to_dense(X_train_t)
    X_test_t = to_dense(X_test_t)
    feature_names = np.array(feature_names)

    preferred = filter_shortlists.get(dataset_name, [])
    preferred = [feat for feat in preferred if feat in set(feature_names)]

    if len(preferred) >= 8:
        idx = [int(np.where(feature_names == f)[0][0]) for f in preferred]
        X_train_pool = X_train_t[:, idx]
        X_test_pool = X_test_t[:, idx]
        pool_features = feature_names[idx]
    else:
        X_train_pool = X_train_t
        X_test_pool = X_test_t
        pool_features = feature_names

    prepared[dataset_name] = {
        'X_train': X_train_pool,
        'X_test': X_test_pool,
        'y_train': y_train,
        'y_test': y_test,
        'feature_names': pool_features,
    }

    print(f"{dataset_name}: pool_features={len(pool_features)}")

## Методы отбора
- `RFE(LogisticRegression)`
- `SequentialFeatureSelector(LogisticRegression)`
- L1-регуляризация (`LogisticRegression`, `penalty='l1'`)
- `RandomForest` feature importance
- permutation importance

TODO: поэкспериментируйте с `n_features_to_select` и сравните стабильность топа.

In [ ]:
ranking_rows = []

for dataset_name, bundle in prepared.items():
    X_train = bundle['X_train']
    X_test = bundle['X_test']
    y_train = bundle['y_train']
    y_test = bundle['y_test']
    feature_names = bundle['feature_names'].tolist()

    n_features_to_select = min(10, max(4, X_train.shape[1] // 2))

    base_lr = LogisticRegression(
        max_iter=2500,
        class_weight='balanced',
        random_state=SEED,
        solver='liblinear',
    )

    # 1) RFE
    rfe = RFE(estimator=base_lr, n_features_to_select=n_features_to_select)
    rfe.fit(X_train, y_train)
    rfe_scores = 1.0 / rfe.ranking_
    append_ranking_rows(ranking_rows, dataset_name, 'rfe', feature_names, rfe_scores)

    # 2) SequentialFeatureSelector
    sfs = SequentialFeatureSelector(
        estimator=base_lr,
        n_features_to_select=n_features_to_select,
        direction='forward',
        scoring='f1',
        cv=4,
        n_jobs=-1,
    )
    sfs.fit(X_train, y_train)
    sfs_scores = sfs.get_support().astype(float)
    append_ranking_rows(ranking_rows, dataset_name, 'sfs_forward', feature_names, sfs_scores)

    # 3) L1 coefficients
    l1_model = LogisticRegression(
        penalty='l1',
        solver='liblinear',
        max_iter=3000,
        class_weight='balanced',
        random_state=SEED,
    )
    l1_model.fit(X_train, y_train)
    l1_scores = np.abs(l1_model.coef_[0])
    append_ranking_rows(ranking_rows, dataset_name, 'l1_logreg', feature_names, l1_scores)

    # 4) RandomForest feature importance
    rf = RandomForestClassifier(
        n_estimators=300,
        random_state=SEED,
        class_weight='balanced_subsample',
        n_jobs=-1,
    )
    rf.fit(X_train, y_train)
    rf_scores = rf.feature_importances_
    append_ranking_rows(ranking_rows, dataset_name, 'rf_importance', feature_names, rf_scores)

    # 5) Permutation importance
    perm = permutation_importance(
        estimator=rf,
        X=X_test,
        y=y_test,
        n_repeats=8,
        random_state=SEED,
        scoring='f1',
        n_jobs=-1,
    )
    perm_scores = np.maximum(perm.importances_mean, 0.0)
    append_ranking_rows(ranking_rows, dataset_name, 'permutation', feature_names, perm_scores)

feature_ranking = (
    pd.DataFrame(ranking_rows)
    .sort_values(['dataset', 'method', 'rank'])
    .reset_index(drop=True)
)
feature_ranking.head(20)

## Формирование 2-3 кандидатных наборов признаков
- `set_A_wrapper` на основе RFE/SFS/L1
- `set_B_tree` на основе RF/permutation
- `set_C_hybrid` как гибрид/пересечение

In [ ]:
feature_sets = {}

for dataset_name in DATASETS:
    set_a = build_shortlist(
        feature_ranking,
        dataset_name,
        methods=['rfe', 'sfs_forward', 'l1_logreg'],
        top_n=10,
    )
    set_b = build_shortlist(
        feature_ranking,
        dataset_name,
        methods=['rf_importance', 'permutation'],
        top_n=10,
    )

    inter = [f for f in set_a if f in set_b]
    if len(inter) < 8:
        for feat in set_a + set_b:
            if feat not in inter:
                inter.append(feat)
            if len(inter) >= 10:
                break

    feature_sets[dataset_name] = {
        'set_A_wrapper': set_a,
        'set_B_tree': set_b,
        'set_C_hybrid': inter[:10],
    }

feature_sets

## Самостоятельное изучение по ходу работы

### Что изучено в этом шаге
- TODO (обязательно): 3-5 предложений о методах/подходах, которые вы изучили именно на этом этапе.
- Укажите, почему эти идеи важны для текущего шага ноутбука.

### Сравнение с альтернативами
- TODO (обязательно): минимум одно сравнение с альтернативным методом/подходом.
- Формат: "когда лучше метод A, когда лучше метод B, и почему".

### Источники
- TODO (обязательно): минимум 1-2 источника (URL и/или `study-notes/*.md`).
- Если объяснение не помещается в ноутбук: вынесите разбор в `study-notes/*.md`
  и оставьте здесь явную ссылку на файл.

### Глоссарий незнакомых терминов (обязательно)
- TODO (обязательно): по ходу этого шага добавьте минимум 2-3 новых термина в `study-notes/glossary.md`.
- Для каждого термина укажите: простое объяснение, где встретился термин в ноутбуке, источник.
- В конце блока напишите строку: `Глоссарий обновлен: <термин_1>, <термин_2>, ...`.


In [ ]:
feature_ranking_path = OUTPUT_DIR / 'feature_ranking_wrapper_embedded.csv'
feature_sets_path = OUTPUT_DIR / 'feature_sets_wrapper_embedded.json'

feature_ranking.to_csv(feature_ranking_path, index=False)
with open(feature_sets_path, 'w', encoding='utf-8') as f:
    json.dump(feature_sets, f, ensure_ascii=False, indent=2)

print(f'Saved: {feature_ranking_path}')
print(f'Saved: {feature_sets_path}')

## Контрольные точки
1. В `feature_ranking` есть строки для 5 методов.
2. Для каждого датасета сформированы `set_A_wrapper`, `set_B_tree`, `set_C_hybrid`.
3. В каждом наборе минимум 5 признаков.

In [ ]:
required_columns = {'dataset', 'method', 'feature', 'score', 'rank'}
assert required_columns.issubset(feature_ranking.columns)

for dataset_name in DATASETS:
    subset = feature_ranking[feature_ranking['dataset'] == dataset_name]
    assert subset['method'].nunique() == 5

    sets = feature_sets[dataset_name]
    for set_name in ['set_A_wrapper', 'set_B_tree', 'set_C_hybrid']:
        assert len(sets[set_name]) >= 5

print('Проверки пройдены успешно.')

## Типичные ошибки
- Слишком большой `n_features_to_select` при малом числе признаков в пуле.
- Неправильная интерпретация L1: нулевые коэффициенты не должны попадать в топ.
- Сравнение permutation importance без фиксированного `random_state`.

## Финальные выводы (заполните)
TODO:
1. Какие признаки чаще пересекаются между wrapper и embedded подходами?
2. Где методы дали заметно разные топы и как это объяснить?
3. Какой набор признаков вы возьмете в следующий ноутбук как основной кандидат?

## Обязательные самостоятельные задания (без образца в solutions)

Ниже задания, которые отсутствуют в `solutions` и обязательны к сдаче.
До заполнения этих блоков ноутбук должен останавливаться с `NotImplementedError`.

### Задание 1. Pairwise agreement matrix для методов

**Контракт задания**

Входные переменные:
- `feature_ranking`, список методов `['rfe', 'sfs_forward', 'l1_logreg', 'rf_importance', 'permutation']`.

Ожидаемый выход:
- `method_agreement_long` с колонками:
  `dataset`, `method_a`, `method_b`, `top_k`, `overlap_count`, `jaccard`.

In [ ]:
required_columns_task1 = [
    'dataset',
    'method_a',
    'method_b',
    'top_k',
    'overlap_count',
    'jaccard',
]
method_agreement_long = pd.DataFrame(columns=required_columns_task1)
assert set(required_columns_task1).issubset(method_agreement_long.columns)

top_k = 10
methods_for_agreement = ['rfe', 'sfs_forward', 'l1_logreg', 'rf_importance', 'permutation']

# TODO(обязательно):
# 1) Для каждого dataset получите top_k признаков для каждого метода.
# 2) Для каждой пары методов посчитайте overlap_count и jaccard.
# 3) Заполните method_agreement_long.


# Обязательный подпункт (методическая рефлексия):
# - До снятия intentional-stop добавьте в релевантный narrative-блок
#   или в `study-notes/*.md` короткое описание: что вы изучили,
#   с чем сравнили, на какие источники опирались.
# - Обновите `study-notes/glossary.md`: добавьте 2-3 термина, встретившихся в этом задании,
#   и укажите их простые объяснения + источники.

raise NotImplementedError(
    'Задание 1 не завершено: заполните method_agreement_long по required_columns_task1.'
)

### Задание 2. Стабильность отбора по random_state/ресемплингу

**Контракт задания**

Входные переменные:
- `prepared`, методы отбора (минимум один wrapper и один embedded).

Ожидаемый выход:
- `selection_stability` с колонками:
  `dataset`, `method`, `feature`, `selected_count`, `total_runs`, `stability_rate`.

In [ ]:
required_columns_task2 = [
    'dataset',
    'method',
    'feature',
    'selected_count',
    'total_runs',
    'stability_rate',
]
selection_stability = pd.DataFrame(columns=required_columns_task2)
assert set(required_columns_task2).issubset(selection_stability.columns)

random_states = [11, 42, 101]

# TODO(обязательно):
# 1) Повторите отбор признаков для нескольких random_state.
# 2) Для каждого feature посчитайте selected_count и stability_rate.
# 3) Заполните selection_stability.


# Обязательный подпункт (методическая рефлексия):
# - До снятия intentional-stop добавьте в релевантный narrative-блок
#   или в `study-notes/*.md` короткое описание: что вы изучили,
#   с чем сравнили, на какие источники опирались.
# - Обновите `study-notes/glossary.md`: добавьте 2-3 термина, встретившихся в этом задании,
#   и укажите их простые объяснения + источники.

raise NotImplementedError(
    'Задание 2 не завершено: заполните selection_stability по required_columns_task2.'
)

### Задание 3. Формирование `set_D_robust` и экспорт

**Контракт задания**

Входные переменные:
- `method_agreement_long`, `selection_stability`, `feature_sets`.

Ожидаемый выход:
- обновленный `feature_sets[dataset]['set_D_robust']`;
- файлы `outputs/method_agreement_long.csv`, `outputs/selection_stability.csv`.

In [ ]:
method_agreement_path = OUTPUT_DIR / 'method_agreement_long.csv'
selection_stability_path = OUTPUT_DIR / 'selection_stability.csv'

required_columns_task1 = {'dataset', 'method_a', 'method_b', 'top_k', 'overlap_count', 'jaccard'}
required_columns_task2 = {'dataset', 'method', 'feature', 'selected_count', 'total_runs', 'stability_rate'}

# TODO(обязательно):
# 1) Убедитесь, что method_agreement_long и selection_stability содержат required columns.
# 2) Сформируйте set_D_robust (например, по порогу stability_rate >= 0.6).
# 3) Добавьте set_D_robust в feature_sets[dataset].
# 4) Сохраните два CSV-файла по указанным путям.


# Обязательный подпункт (методическая рефлексия):
# - До снятия intentional-stop добавьте в релевантный narrative-блок
#   или в `study-notes/*.md` короткое описание: что вы изучили,
#   с чем сравнили, на какие источники опирались.
# - Обновите `study-notes/glossary.md`: добавьте 2-3 термина, встретившихся в этом задании,
#   и укажите их простые объяснения + источники.

raise NotImplementedError(
    'Задание 3 не завершено: сформируйте set_D_robust и сохраните method_agreement_long/selection_stability.'
)